# Chapter 5 · Grover's Algorithm

## Objectives

1. Understand amplitude amplification as a quantum speedup mechanism.
2. Implement the Grover oracle and the diffusion operator.
3. Verify the optimal iteration condition $k \approx \pi/(4\arcsin(1/\sqrt{N}))$.
4. Analyze the quadratic advantage: $O(\sqrt{N})$ vs. $O(N)$ classical.

---

## 5.1 Mathematical background

Let $|s\rangle = H^{\otimes n}|0\rangle^{\otimes n}$ be the uniform superposition. The Grover operator is:

$$G = D \cdot U_\omega, \quad D = 2|s\rangle\langle s| - I, \quad U_\omega|x\rangle = (-1)^{f(x)}|x\rangle$$

Geometrically, $G$ performs a rotation of angle $2\theta$ in the plane spanned by $|s\rangle$ and $|\omega\rangle$, with $\sin\theta = 1/\sqrt{N}$. After $k$ iterations:

$$G^k|s\rangle = \sin((2k+1)\theta)|\omega\rangle + \cos((2k+1)\theta)|s'\rangle$$

The success probability is maximized when $(2k+1)\theta \approx \pi/2$, i.e. $k_{\text{opt}} \approx \pi/(4\theta)$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Dependencies loaded.')

## 5.2 Building the Grover oracle and the diffuser

In [ ]:
def grover_oracle(n: int, target: int) -> QuantumCircuit:
    """Grover oracle that marks the state |target⟩ with a -1 phase shift.

    Implementation: multi-controlled Z (MCZ) on the target state.

    Parameters
    ----------
    n : int
        Number of qubits.
    target : int
        Index of the state to mark (0 ≤ target < 2^n).
    """
    qc = QuantumCircuit(n, name=f'Oracle({format(target, f"0{n}b")})')
    target_bits = format(target, f'0{n}b')

    # Flip qubits that are '0' in the target (so MCZ gives -(-1))
    for i, bit in enumerate(reversed(target_bits)):
        if bit == '0':
            qc.x(i)

    # MCZ: implemented as H + MCX + H on the most significant qubit
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)

    # Undo the flips
    for i, bit in enumerate(reversed(target_bits)):
        if bit == '0':
            qc.x(i)
    return qc


def grover_diffuser(n: int) -> QuantumCircuit:
    """Diffusion operator D = 2|s⟩⟨s| - I.

    Corresponds to a reflection about the uniform superposition state.
    """
    qc = QuantumCircuit(n, name='Diffuser')
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc


n = 4
target = 11  # = |1011⟩
print('Oracle:')
print(grover_oracle(n, target).draw('text'))
print('\nDiffuser:')
print(grover_diffuser(n).draw('text'))

## 5.3 Full Grover circuit

In [ ]:
def grover_circuit(n: int, target: int, iterations: int | None = None) -> QuantumCircuit:
    """Builds the full circuit for Grover's algorithm.

    Parameters
    ----------
    n : int
        Number of qubits.
    target : int
        Marked item.
    iterations : int, optional
        Number of G iterations. Defaults to the theoretical optimum.
    """
    N = 2 ** n
    if iterations is None:
        theta = np.arcsin(1 / np.sqrt(N))
        iterations = int(np.round(np.pi / (4 * theta)))

    qc = QuantumCircuit(n, n)

    # Initial superposition
    qc.h(range(n))
    qc.barrier()

    # k Grover iterations
    oracle   = grover_oracle(n, target)
    diffuser = grover_diffuser(n)
    for _ in range(iterations):
        qc.compose(oracle,   inplace=True)
        qc.compose(diffuser, inplace=True)
        qc.barrier()

    # Measurement
    qc.measure(range(n), range(n))
    return qc, iterations


n      = 4
target = 11
qc_grover, k_opt = grover_circuit(n, target)

print(f'n={n} qubits, N={2**n}, target={target} ({format(target, f"0{n}b")})')
print(f'Optimal number of iterations: k = {k_opt}')

backend = AerSimulator()
job = backend.run(qc_grover, shots=4096)
counts = job.result().get_counts()

# Results
print('\nTop 5 most frequent results:')
for state, cnt in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    prob = cnt / 4096
    marker = ' ← TARGET' if int(state, 2) == target else ''
    print(f'  |{state}⟩ ({int(state, 2):2d}): {cnt:4d} times  ({prob:.3f}){marker}')

fig = QuantumVisualization.plot_histogram(
    counts, title=f'Grover: n={n}, target={format(target, f"0{n}b")} ({target}), k={k_opt}',
    color='#7ee787'
)
plt.show()

## 5.4 Evolution of success probability with number of iterations

In [ ]:
n = 4
target = 7
N = 2**n

# Analytical calculation
k_range = range(1, 15)
theta = np.arcsin(1 / np.sqrt(N))
prob_analytic = [np.sin((2*k + 1) * theta)**2 for k in k_range]

# Full simulation for each k
prob_sim = []
backend = AerSimulator()
for k in k_range:
    qc, _ = grover_circuit(n, target, iterations=k)
    job = backend.run(qc, shots=2048)
    cnts = job.result().get_counts()
    target_str = format(target, f'0{n}b')
    prob_sim.append(cnts.get(target_str, 0) / 2048)

# Visualization
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), prob_analytic, '-', color='#58a6ff',
        linewidth=2, label='Analytical')
ax.plot(list(k_range), prob_sim, 'o', color='#f78166',
        markersize=7, label='Qiskit simulation')
ax.axvline(int(np.round(np.pi / (4 * theta))),
           linestyle='--', color='#7ee787', alpha=0.7, label='Optimal k')
ax.set_xlabel('Grover iterations (k)')
ax.set_ylabel('Success probability P(target)')
ax.set_title(f'Success probability vs. iterations (n={n}, target={target})')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 5.5 Proposed exercises

1. Implement Grover to search for **two** marked items simultaneously. How does the optimal number of iterations change?

2. For $n=2$, run the algorithm for all possible target choices ($0,1,2,3$) and verify the result.

3. Implement Grover's algorithm to solve the 3-SAT problem with 3 variables. Encode the clause $(x_1 \lor \neg x_2 \lor x_3)$ as an oracle.

4. What happens if more than $k_{\text{opt}}$ iterations are applied? Plot the success probability for $k \in [1, 3k_{\text{opt}}]$.